# Session 2 — CI/CD for ML Systems

**Goal:** Understand how the manual workflow from Session 1 becomes an automated pipeline.

In Session 1 you manually:
1. Ran `04_mlflow_experiment.ipynb` → compared models
2. Ran `05_register_and_serve.ipynb` → deployed the best one

In production, this **must** happen automatically when:
- New training data arrives
- Code is merged to the main branch
- A scheduled retrain job triggers

**The tools:** Declarative Automation Bundles (DABs) + GitHub Actions

In [0]:
%run ../utils/config

## Declarative Automation Bundles
Formerly known as Databricks Asset Bundles, facilitate the adoption of software engineering best practices, including source control, code review, testing, and continuous integration and delivery (CI/CD)

Declarative Automation Bundles include:
-  Infrastructure and workspace configurations
-  Source files
-  Definitionsfor Databricks resources (i.e. Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Dashboards, Model Serving endpoints, MLflow Experiments, and MLflow registered models)
-  Unit tests and integration tests

<img src="../utils/resources/bundles-cicd.png">

Declarative Automation Bundles are typically developed in a local development environment such as VSCode or Cursor.  Databricks also supports [DABs in the Workspace](https://www.databricks.com/blog/announcing-databricks-asset-bundles-now-workspace) which enables you to build, edit and deploy using DABs from within a Databricks workspace.  We will use DABs in the Workspace in this workshop so we won't need a local dev environment.

## Navigate the Declarative Automation Bundle

1. Navigate to `../bundle` in the **Workspace Navigator**
2. Open `databricks.yml` — the root definition file - in another browser tab.
3. Open `bundle/resources/retrain_job.yml` — defines the Lakeflow Job that automates the training pipeline from Session 1
4.  Click the rocket icon (![](../utils/resources/DeploymentIcon.png)) in the left nav bar.

This switches to a the bundle editor.
1.  Confirm the Target is set to `Dev`

## Edit the Declarative Automation Bundle
Each person in this workshop will run the job targeting their own personal schema.  To do this, you will configure a target override variable.

1.  Run the cell below.  This will generate the JSON configuration you need.
2.  Copy the JSON generated by the next cell.
3.  Switch back to the bundle editor.
4.  Select **Configure variable overrides** from the kebab menu next to `Dev`.
![](../utils/resources/ConfigureVariableOverrides.png)
5.  Paste the JSON you copied into the overrides file, replacing whatever is currently there.

In [0]:
print('{')
print(f'    "schema": "{schema}"')
print('}')

## The Retrain Pipeline


1. Deploy the bundle by clicking **Deploy**, then **Deploy** again
2. Once the deploy finishes successfully, click the **Run job** icon next to the `workshop_retrain_churn` job

The first three tasks take about 3 min.  The final task, deploying the endpoint, takes an additional 8-10 min.

The retrain job is available in **Workflows → Jobs** as `[dev <your_schema>] workshop_retrain_churn`.
1.  Click on `workspace_retrain_churn` in the bundle editor.  This opens the job in a new tab.
2.  Verify the job finishes successfully.

## The Retrain Pipeline DAG

The retrain job has 4 tasks with dependencies:

```
Task 1: ingest_and_validate
    ↓  (fails if data quality checks fail)
Task 2: train_models
    ↓  (logs 3 MLflow runs, picks best)
Task 3: gate_and_register
    ↓  (fails if F1 drops > 5%)
Task 4: update_endpoint
```

This design ensures:
- **Bad data never triggers training** (Task 1 gate)
- **Worse models are never deployed** (Task 3 gate)
- **Every deployment is traceable** (MLflow + UC lineage)

The full job definition lives at `bundle/resources/retrain_job.yml` in the Workspace Navigator. Open it to see the complete task configuration, cluster settings, and parameter passing between tasks.

## The GitHub Actions Workflow

TODO: Add more discussion on Git actions.

The CI/CD pipeline has two triggers:

**On Pull Request** (`.github/workflows/pr_validation.yml`):
```
push to PR branch → run pytest → databricks bundle validate
```
Prevents merging broken code.

**On Merge to Main** (`.github/workflows/deploy_retrain.yml`):
```
merge to main → bundle deploy → bundle run retrain_job
```
Automatically deploys and retrains after every approved code change.

The PR validation workflow lives at `.github/workflows/pr_validation.yml`. It runs `pytest` and `databricks bundle validate` on every pull request — preventing broken code or invalid bundle configs from reaching main.

## Discussion

- Where should the `DATABRICKS_TOKEN` come from? (Service principal, not a human user)
- What happens if the test fails? (PR is blocked from merging)
- What's the difference between `bundle validate` and `bundle deploy`?

➡️ Next: `02_simulate_drift.ipynb` — inject realistic production drift into the inference log